# Phase 14 — Normalization and Feature Builder Implementation

This notebook implements reusable feature-building cells for model-core training inputs. It keeps execution notebook-first and writes durable validation reports under `reports/` for later pair generation, baseline evaluation, model cards, and export manifests.

The cells are intentionally deterministic and dependency-light. Embedding validation uses a local deterministic hash embedder as a cache-contract probe; later training notebooks may swap in a production embedding model only when the model version, row count, source hash, shape, dtype, finite-value checks, and cache hash still validate.

## Purpose
Document and verify Phase 14 — Normalization and Feature Builder Implementation in the Bisakerja notebook-first training workflow.

## Required input
Use the repository-root training data, artifacts, and reports referenced by this phase.

## Action
Run or review the Phase 14.normalization.feature.builder notebook cells in numeric order, preserving generated evidence under reports/ and artifacts/.

## Expected output
Produce or preserve the phase-specific report and artifact evidence for Phase 14 — Normalization and Feature Builder Implementation.

## Verification
Confirm the notebook has no saved error outputs, no unintended unexecuted production code cells, and matching durable report evidence.

## Step 14.1 — Experience normalization

### Purpose
Normalize observed profile and job experience values into stable bands before feature generation.

### Required input
`legacy/dataset/indotech_job_cleaned.csv`, `legacy/dataset/techtalent_profile_cleaned.csv`, and the allowed values `Fresher`, `1-2 years`, `3-5 years`, `5+ years`, `ENTRY_LEVEL`, `JUNIOR`, `MID_LEVEL`, `SENIOR`, `LEAD`, and `MANAGER`.

### Action
Define a versioned experience mapping, load observed values, normalize them, and fail if any non-empty observed value falls through silently.

### Expected output
A normalized experience map with numeric ranges, ordered bands, and an unknown-observed-value report.

### Verification
The check `experience_no_observed_fallthrough` must pass and the observed unknown list must be empty.


In [1]:
from __future__ import annotations

import csv
import hashlib
import json
import math
import os
import random
import re
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

PHASE_ID = "phase_14_normalization_feature_builder"
SEED = 202614
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "TODOS.md").exists() and (candidate / "training" / "notebooks").exists():
            return candidate
    raise RuntimeError("Repository root not found. Start notebook inside bisakerja-model repository.")


REPO_ROOT = find_repo_root()
REPORTS_DIR = REPO_ROOT / "reports"
REPORTS_DIR.mkdir(exist_ok=True)

JOBS_PATH = REPO_ROOT / "legacy/dataset/indotech_job_cleaned.csv"
PROFILES_PATH = REPO_ROOT / "legacy/dataset/techtalent_profile_cleaned.csv"
FEATURE_SCHEMA_VERSION = "normalization-feature-builder-v1"
EMBEDDING_MODEL_VERSION = "local-hash-embedding-v1"
EMBEDDING_DIM = 16
ALLOWED_LANGUAGES = {"ID", "EN", "MIXED", "UNKNOWN"}


def rel(path: Path) -> str:
    return path.resolve().relative_to(REPO_ROOT).as_posix()


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def sha256_json(value: Any) -> str:
    payload = json.dumps(value, sort_keys=True, ensure_ascii=False, separators=(",", ":")).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()


def read_csv_rows(path: Path) -> list[dict[str, str]]:
    with path.open(newline="", encoding="utf-8") as handle:
        return [dict(row) for row in csv.DictReader(handle)]


def write_json_report(name: str, payload: Any) -> Path:
    path = REPORTS_DIR / name
    path.write_text(json.dumps(payload, indent=2, sort_keys=True, ensure_ascii=False) + "\n")
    return path


jobs = read_csv_rows(JOBS_PATH)
profiles = read_csv_rows(PROFILES_PATH)

EXPERIENCE_NORMALIZATION = {
    "schema_version": FEATURE_SCHEMA_VERSION,
    "mapping_version": "experience-normalization-v1",
    "values": {
        "Fresher": {"band": "FRESHER", "min_years": 0.0, "max_years": 0.5, "order": 0},
        "1-2 years": {"band": "JUNIOR", "min_years": 1.0, "max_years": 2.0, "order": 1},
        "3-5 years": {"band": "MID_LEVEL", "min_years": 3.0, "max_years": 5.0, "order": 2},
        "5+ years": {"band": "SENIOR_PLUS", "min_years": 5.0, "max_years": None, "order": 3},
        "ENTRY_LEVEL": {"band": "ENTRY_LEVEL", "min_years": 0.0, "max_years": 1.0, "order": 0},
        "JUNIOR": {"band": "JUNIOR", "min_years": 1.0, "max_years": 2.0, "order": 1},
        "MID_LEVEL": {"band": "MID_LEVEL", "min_years": 3.0, "max_years": 5.0, "order": 2},
        "SENIOR": {"band": "SENIOR", "min_years": 5.0, "max_years": 8.0, "order": 3},
        "LEAD": {"band": "LEAD", "min_years": 7.0, "max_years": None, "order": 4},
        "MANAGER": {"band": "MANAGER", "min_years": 7.0, "max_years": None, "order": 4},
    },
}


def normalize_experience(value: Any) -> dict[str, Any]:
    raw = "" if value is None else str(value).strip()
    if not raw:
        return {"raw": raw, "normalized": "UNKNOWN", "is_unknown": True, "reason": "empty"}
    entry = EXPERIENCE_NORMALIZATION["values"].get(raw)
    if entry is None:
        return {"raw": raw, "normalized": "UNKNOWN", "is_unknown": True, "reason": "unmapped_observed_value"}
    return {"raw": raw, "normalized": entry["band"], "is_unknown": False, **entry}

observed_experience_values = {
    "jobs.experience_level": sorted({row.get("experience_level", "").strip() for row in jobs if row.get("experience_level", "").strip()}),
    "profiles.Experience": sorted({row.get("Experience", "").strip() for row in profiles if row.get("Experience", "").strip()}),
}
unknown_observed_experience = {
    source: [value for value in values if normalize_experience(value)["is_unknown"]]
    for source, values in observed_experience_values.items()
}
experience_no_observed_fallthrough = all(not values for values in unknown_observed_experience.values())
if not experience_no_observed_fallthrough:
    raise ValueError(f"Unmapped observed experience values: {unknown_observed_experience}")

experience_report = {
    "schema_version": FEATURE_SCHEMA_VERSION,
    "mapping_version": EXPERIENCE_NORMALIZATION["mapping_version"],
    "observed_values": observed_experience_values,
    "unknown_observed_values": unknown_observed_experience,
    "passed": experience_no_observed_fallthrough,
}
experience_report


{'schema_version': 'normalization-feature-builder-v1',
 'mapping_version': 'experience-normalization-v1',
 'observed_values': {'jobs.experience_level': ['ENTRY_LEVEL',
   'JUNIOR',
   'LEAD',
   'MID_LEVEL',
   'SENIOR'],
  'profiles.Experience': ['1-2 years', '3-5 years', '5+ years', 'Fresher']},
 'unknown_observed_values': {'jobs.experience_level': [],
  'profiles.Experience': []},
 'passed': True}

## Step 14.2 — Skill alias normalization

### Purpose
Normalize skill tokens with a versioned alias table so skill overlap and missing-skill signals are reproducible.

### Required input
Profile `Skills`, profile `Required_Skills`, job `skills_clean`, job `requirements_concat`, Indonesian/English aliases, framework variants, and punctuation-heavy tokens.

### Action
Parse skill-like values, clean punctuation, apply aliases, preserve unknown tokens for reporting, and calculate unknown-skill token rates by source.

### Expected output
Normalized skill lists and a skill-quality report with alias version, observed tokens, unknown tokens, empty-skill rates, and top unknown examples.

### Verification
Known aliases map to canonical values and unknown token reporting is non-silent.


In [2]:
SKILL_ALIAS_VERSION = "skill-alias-normalization-v1"
SKILL_ALIASES = {
    "js": "javascript",
    "javascript": "javascript",
    "typescript": "typescript",
    "ts": "typescript",
    "reactjs": "react",
    "react.js": "react",
    "react js": "react",
    "react": "react",
    "nextjs": "next.js",
    "next.js": "next.js",
    "nodejs": "node.js",
    "node.js": "node.js",
    "node js": "node.js",
    "vuejs": "vue.js",
    "vue.js": "vue.js",
    "python": "python",
    "py": "python",
    "java": "java",
    "golang": "go",
    "go": "go",
    "laravel": "laravel",
    "php": "php",
    "sql": "sql",
    "mysql": "mysql",
    "postgres": "postgresql",
    "postgresql": "postgresql",
    "mongo": "mongodb",
    "mongodb": "mongodb",
    "aws": "aws",
    "gcp": "gcp",
    "google cloud": "gcp",
    "docker": "docker",
    "kubernetes": "kubernetes",
    "k8s": "kubernetes",
    "machine learning": "machine learning",
    "ml": "machine learning",
    "ai": "artificial intelligence",
    "artificial intelligence": "artificial intelligence",
    "analisis data": "data analysis",
    "data analysis": "data analysis",
    "data analytics": "data analysis",
    "manajemen proyek": "project management",
    "project management": "project management",
    "komunikasi": "communication",
    "communication": "communication",
    "kepemimpinan": "leadership",
    "leadership": "leadership",
    "pemecahan masalah": "problem solving",
    "problem solving": "problem solving",
    "ui/ux": "ui/ux",
    "figma": "figma",
}


def clean_skill_token(token: Any) -> str:
    text = "" if token is None else str(token).lower().strip()
    text = text.replace("&", " and ")
    text = re.sub(r"[\[\]\{\}\(\)'\"`]+", " ", text)
    text = re.sub(r"\s*/\s*", "/", text)
    text = re.sub(r"[^a-z0-9+#./\-\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip(" -.,;")
    return text


def split_skill_tokens(value: Any) -> list[str]:
    if value is None:
        return []
    text = str(value).strip()
    if not text:
        return []
    parts = re.split(r"[,;|\n]+", text)
    return [clean_skill_token(part) for part in parts if clean_skill_token(part)]


def normalize_skill_token(token: Any) -> dict[str, Any]:
    cleaned = clean_skill_token(token)
    if not cleaned:
        return {"raw": token, "cleaned": cleaned, "canonical": "UNKNOWN", "is_unknown": True}
    canonical = SKILL_ALIASES.get(cleaned, cleaned)
    known = cleaned in SKILL_ALIASES
    return {"raw": token, "cleaned": cleaned, "canonical": canonical, "is_unknown": not known}


def normalize_skill_list(value: Any) -> list[dict[str, Any]]:
    seen = set()
    normalized = []
    for token in split_skill_tokens(value):
        item = normalize_skill_token(token)
        canonical = item["canonical"]
        if canonical not in seen:
            normalized.append(item)
            seen.add(canonical)
    return normalized

skill_sources = {
    "jobs.skills_clean": [row.get("skills_clean", "") for row in jobs],
    "jobs.requirements_concat": [row.get("requirements_concat", "") for row in jobs],
    "profiles.Skills": [row.get("Skills", "") for row in profiles],
    "profiles.Required_Skills": [row.get("Required_Skills", "") for row in profiles],
}

skill_quality: dict[str, Any] = {}
for source, values in skill_sources.items():
    token_count = 0
    unknown_count = 0
    empty_rows = 0
    unknown_examples = Counter()
    canonical_examples = Counter()
    for value in values:
        normalized = normalize_skill_list(value)
        if not normalized:
            empty_rows += 1
        for item in normalized:
            token_count += 1
            canonical_examples[item["canonical"]] += 1
            if item["is_unknown"]:
                unknown_count += 1
                unknown_examples[item["cleaned"]] += 1
    row_count = len(values)
    skill_quality[source] = {
        "row_count": row_count,
        "token_count": token_count,
        "empty_skill_rows": empty_rows,
        "empty_skill_rate": empty_rows / row_count if row_count else 0.0,
        "unknown_skill_tokens": unknown_count,
        "unknown_skill_token_rate": unknown_count / token_count if token_count else 0.0,
        "top_unknown_tokens": unknown_examples.most_common(20),
        "top_canonical_tokens": canonical_examples.most_common(20),
    }

alias_verification = {
    "ReactJS": normalize_skill_token("ReactJS")["canonical"] == "react",
    "analisis data": normalize_skill_token("analisis data")["canonical"] == "data analysis",
    "k8s": normalize_skill_token("k8s")["canonical"] == "kubernetes",
}
if not all(alias_verification.values()):
    raise ValueError(f"Skill alias verification failed: {alias_verification}")

skill_report = {
    "schema_version": FEATURE_SCHEMA_VERSION,
    "alias_version": SKILL_ALIAS_VERSION,
    "alias_count": len(SKILL_ALIASES),
    "quality": skill_quality,
    "alias_verification": alias_verification,
}
skill_report


{'schema_version': 'normalization-feature-builder-v1',
 'alias_version': 'skill-alias-normalization-v1',
 'alias_count': 51,
 'quality': {'jobs.skills_clean': {'row_count': 2073,
   'token_count': 7298,
   'empty_skill_rows': 0,
   'empty_skill_rate': 0.0,
   'unknown_skill_tokens': 5725,
   'unknown_skill_token_rate': 0.7844614963003562,
   'top_unknown_tokens': [('effective communication', 334),
    ('programming', 280),
    ('expires soon', 189),
    ('akan segera berakhir', 189),
    ('ci/cd', 72),
    ('it support', 64),
    ('rest api', 59),
    ('ci', 58),
    ('git', 58),
    ('cd', 56),
    ('linux', 51),
    ('quality assurance', 48),
    ('html', 47),
    ('network security', 47),
    ('css', 46),
    ('business analysis', 43),
    ('ux design', 40),
    ('microsoft sql server', 39),
    ('teamwork', 39),
    ('troubleshooting', 39)],
   'top_canonical_tokens': [('effective communication', 334),
    ('programming', 280),
    ('expires soon', 189),
    ('akan segera berakhir'

## Step 14.3 — Language normalization

### Purpose
Normalize language signals into the supported slices `ID`, `EN`, `MIXED`, and `UNKNOWN`.

### Required input
Job `language_signal`, available profile/job text, and simple Indonesian/English lexical evidence for fallback classification.

### Action
Map direct language labels, infer text-only fallback when needed, and publish slice-level counts and unknown rates.

### Expected output
Language slice report with counts, unknown rate, unsupported raw values, and deterministic fallback policy.

### Verification
All normalized labels are one of `ID`, `EN`, `MIXED`, or `UNKNOWN`; unsupported observed labels are reported.


In [3]:
LANGUAGE_NORMALIZATION_VERSION = "language-normalization-v1"
ID_MARKERS = {"dan", "yang", "dengan", "untuk", "pengalaman", "keahlian", "minimal", "tahun", "kerja", "kemampuan"}
EN_MARKERS = {"and", "with", "for", "experience", "skills", "minimum", "years", "work", "ability", "requirements"}


def normalize_language(value: Any, text: str = "") -> dict[str, Any]:
    raw = "" if value is None else str(value).strip().upper()
    if raw in ALLOWED_LANGUAGES:
        return {"raw": raw, "normalized": raw, "reason": "direct_label"}
    tokens = set(re.findall(r"[a-zA-Z]+", text.lower()))
    id_hits = len(tokens & ID_MARKERS)
    en_hits = len(tokens & EN_MARKERS)
    if id_hits and en_hits:
        label = "MIXED"
    elif id_hits:
        label = "ID"
    elif en_hits:
        label = "EN"
    else:
        label = "UNKNOWN"
    return {"raw": raw, "normalized": label, "reason": "text_fallback", "id_hits": id_hits, "en_hits": en_hits}

language_rows = []
for row in jobs:
    text = " ".join([row.get("title", ""), row.get("description", ""), row.get("requirements_concat", "")])
    language_rows.append(normalize_language(row.get("language_signal", ""), text))

language_counts = Counter(item["normalized"] for item in language_rows)
unsupported_raw_language = sorted({item["raw"] for item in language_rows if item["raw"] and item["raw"] not in ALLOWED_LANGUAGES})
language_report = {
    "schema_version": FEATURE_SCHEMA_VERSION,
    "normalization_version": LANGUAGE_NORMALIZATION_VERSION,
    "allowed_languages": sorted(ALLOWED_LANGUAGES),
    "row_count": len(language_rows),
    "counts": dict(sorted(language_counts.items())),
    "unknown_language_rate": language_counts.get("UNKNOWN", 0) / len(language_rows) if language_rows else 0.0,
    "unsupported_raw_values": unsupported_raw_language,
    "passed": not unsupported_raw_language and set(language_counts).issubset(ALLOWED_LANGUAGES),
}
if not language_report["passed"]:
    raise ValueError(f"Unsupported language normalization values: {language_report}")
language_report


{'schema_version': 'normalization-feature-builder-v1',
 'normalization_version': 'language-normalization-v1',
 'allowed_languages': ['EN', 'ID', 'MIXED', 'UNKNOWN'],
 'row_count': 2073,
 'counts': {'EN': 1426, 'ID': 22, 'MIXED': 23, 'UNKNOWN': 602},
 'unknown_language_rate': 0.2904003859141341,
 'unsupported_raw_values': [],
 'passed': True}

## Step 14.4 — Text feature builders

### Purpose
Build deterministic text fields for profile/CV text, job text, role text, requirement text, and section-aware CV text.

### Required input
Profile skill/project/education/experience/role columns, job title/description/requirements/skills/category columns, and future parsed CV section records.

### Action
Compose labeled text segments, trim whitespace, skip empty segments, and report empty-text rates by feature type.

### Expected output
Reusable text-builder functions and feature-quality metrics for profile, job, role, requirement, and CV-section text.

### Verification
Fixture rows produce non-empty text where source data exists; empty text rates are exported for later gate checks.


In [4]:
def compact_text(value: Any) -> str:
    text = "" if value is None else str(value)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def labeled_segment(label: str, value: Any) -> str:
    text = compact_text(value)
    return f"{label}: {text}" if text else ""


def join_segments(*segments: str) -> str:
    return " | ".join(segment for segment in segments if segment)


def build_profile_text(row: dict[str, Any]) -> str:
    return join_segments(
        labeled_segment("target_role", row.get("Job_Role")),
        labeled_segment("experience", row.get("Experience")),
        labeled_segment("skills", row.get("Skills")),
        labeled_segment("required_skills", row.get("Required_Skills")),
        labeled_segment("projects", row.get("Projects")),
        labeled_segment("education", row.get("Education")),
    )


def build_job_text(row: dict[str, Any]) -> str:
    return join_segments(
        labeled_segment("title", row.get("title")),
        labeled_segment("normalized_title", row.get("normalized_title")),
        labeled_segment("category", row.get("category")),
        labeled_segment("experience", row.get("experience_level")),
        labeled_segment("skills", row.get("skills_clean")),
        labeled_segment("requirements", row.get("requirements_concat")),
        labeled_segment("description", row.get("description")),
    )


def build_role_text(row: dict[str, Any]) -> str:
    return join_segments(labeled_segment("role", row.get("Job_Role") or row.get("title")), labeled_segment("category", row.get("category")))


def build_requirement_text(row: dict[str, Any]) -> str:
    return join_segments(labeled_segment("requirements", row.get("requirements_concat") or row.get("Required_Skills")), labeled_segment("experience", row.get("experience_level") or row.get("Experience")))


def build_section_aware_cv_text(sections: list[dict[str, Any]]) -> str:
    ordered = sorted(sections, key=lambda item: str(item.get("section_name", "")))
    return join_segments(*[labeled_segment(str(item.get("section_name", "section")).lower(), item.get("section_text", "")) for item in ordered])

profile_texts = [build_profile_text(row) for row in profiles]
job_texts = [build_job_text(row) for row in jobs]
role_texts = [build_role_text(row) for row in profiles[:1000]] + [build_role_text(row) for row in jobs]
requirement_texts = [build_requirement_text(row) for row in jobs] + [build_requirement_text(row) for row in profiles[:1000]]
cv_fixture_text = build_section_aware_cv_text([
    {"section_name": "summary", "section_text": "Backend engineer with Python and SQL experience."},
    {"section_name": "skills", "section_text": "Python, PostgreSQL, Docker"},
])


def empty_rate(values: list[str]) -> dict[str, Any]:
    empty = sum(1 for value in values if not compact_text(value))
    return {"row_count": len(values), "empty_count": empty, "empty_rate": empty / len(values) if values else 0.0}

text_quality = {
    "profile_text": empty_rate(profile_texts),
    "job_text": empty_rate(job_texts),
    "role_text": empty_rate(role_texts),
    "requirement_text": empty_rate(requirement_texts),
    "section_aware_cv_fixture": {"row_count": 1, "empty_count": 0 if cv_fixture_text else 1, "empty_rate": 0.0 if cv_fixture_text else 1.0},
}

if not profile_texts[0] or not job_texts[0] or not cv_fixture_text:
    raise ValueError("Text builder fixture verification failed")

text_report = {
    "schema_version": FEATURE_SCHEMA_VERSION,
    "text_builder_version": "text-builder-v1",
    "quality": text_quality,
    "sample_hashes": {
        "profile_text_first": sha256_json(profile_texts[0]),
        "job_text_first": sha256_json(job_texts[0]),
        "section_aware_cv_fixture": sha256_json(cv_fixture_text),
    },
}
text_report


{'schema_version': 'normalization-feature-builder-v1',
 'text_builder_version': 'text-builder-v1',
 'quality': {'profile_text': {'row_count': 69929,
   'empty_count': 0,
   'empty_rate': 0.0},
  'job_text': {'row_count': 2073, 'empty_count': 0, 'empty_rate': 0.0},
  'role_text': {'row_count': 3073, 'empty_count': 0, 'empty_rate': 0.0},
  'requirement_text': {'row_count': 3073, 'empty_count': 0, 'empty_rate': 0.0},
  'section_aware_cv_fixture': {'row_count': 1,
   'empty_count': 0,
   'empty_rate': 0.0}},
 'sample_hashes': {'profile_text_first': 'ed8f28a14c9e677c175932a80f2d611268058a1cf5764f4b0e7d159d931f3234',
  'job_text_first': 'efe74dd60623defcc7968f5b8b3525835f98502e74970e06253fc885a158c1dc',
  'section_aware_cv_fixture': '604571f92f1e9739d796c8d8a8c22c79897956cc4e5db1f8ce31172f962784e8'}}

## Step 14.5 — Embedding generation and cache validation

### Purpose
Validate embedding-cache safety before expensive model embeddings are introduced.

### Required input
Built text features, embedding model version, expected shape, dtype, source row count, source text hash, and existing cache metadata when present.

### Action
Generate deterministic local hash embeddings for fixture rows, write a manifest, validate finite numeric values, and reject cache reuse when model version, row count, source hash, shape, dtype, or embedding hash mismatches.

### Expected output
`reports/phase_14_embedding_manifest.json`, `reports/phase_14_feature_quality_report.json`, and `reports/phase_14_normalization_feature_builder.json`.

### Verification
Cache validation passes for the generated manifest and fails for intentionally mismatched metadata.


In [5]:
def hash_embedding(text: str, dim: int = EMBEDDING_DIM) -> list[float]:
    values = []
    for index in range(dim):
        digest = hashlib.sha256(f"{EMBEDDING_MODEL_VERSION}:{index}:{text}".encode("utf-8")).digest()
        integer = int.from_bytes(digest[:8], "big", signed=False)
        values.append((integer / ((1 << 64) - 1)) * 2.0 - 1.0)
    norm = math.sqrt(sum(value * value for value in values)) or 1.0
    return [round(value / norm, 8) for value in values]

embedding_texts = profile_texts[:25] + job_texts[:25]
source_text_hash = sha256_json(embedding_texts)
embeddings = [hash_embedding(text) for text in embedding_texts]
embedding_shape = [len(embeddings), EMBEDDING_DIM]
embedding_dtype = "float64-json"
finite_values = all(math.isfinite(value) for row in embeddings for value in row)
embedding_hash = sha256_json(embeddings)

embedding_manifest = {
    "schema_version": FEATURE_SCHEMA_VERSION,
    "embedding_model_version": EMBEDDING_MODEL_VERSION,
    "row_count": len(embedding_texts),
    "source_text_hash": source_text_hash,
    "shape": embedding_shape,
    "dtype": embedding_dtype,
    "finite_values": finite_values,
    "embedding_hash": embedding_hash,
    "created_at": datetime.now(timezone.utc).isoformat(),
}


def validate_embedding_cache(manifest: dict[str, Any], expected: dict[str, Any]) -> dict[str, Any]:
    checks = {
        "model_version": manifest.get("embedding_model_version") == expected.get("embedding_model_version"),
        "row_count": manifest.get("row_count") == expected.get("row_count"),
        "source_text_hash": manifest.get("source_text_hash") == expected.get("source_text_hash"),
        "shape": manifest.get("shape") == expected.get("shape"),
        "dtype": manifest.get("dtype") == expected.get("dtype"),
        "finite_values": bool(manifest.get("finite_values")),
        "embedding_hash": manifest.get("embedding_hash") == expected.get("embedding_hash"),
    }
    return {"passed": all(checks.values()), "checks": checks}

expected_cache = dict(embedding_manifest)
cache_validation = validate_embedding_cache(embedding_manifest, expected_cache)
mismatch_expected = dict(expected_cache)
mismatch_expected["embedding_model_version"] = "different-model-version"
mismatch_validation = validate_embedding_cache(embedding_manifest, mismatch_expected)

if not cache_validation["passed"]:
    raise ValueError(f"Generated embedding cache failed validation: {cache_validation}")
if mismatch_validation["passed"]:
    raise ValueError("Embedding cache mismatch verification failed")

feature_quality_report = {
    "schema_version": FEATURE_SCHEMA_VERSION,
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "source_files": {
        "jobs": {"path": rel(JOBS_PATH), "sha256": sha256_file(JOBS_PATH), "row_count": len(jobs)},
        "profiles": {"path": rel(PROFILES_PATH), "sha256": sha256_file(PROFILES_PATH), "row_count": len(profiles)},
    },
    "unknown_language_rate": language_report["unknown_language_rate"],
    "unknown_experience": unknown_observed_experience,
    "unknown_experience_observed_value_count": sum(len(values) for values in unknown_observed_experience.values()),
    "empty_skills": {source: {"empty_skill_rows": data["empty_skill_rows"], "empty_skill_rate": data["empty_skill_rate"]} for source, data in skill_quality.items()},
    "empty_text": text_quality,
    "unknown_skill_token_rates": {source: data["unknown_skill_token_rate"] for source, data in skill_quality.items()},
}

phase_report = {
    "phase_id": PHASE_ID,
    "status": "complete",
    "schema_version": FEATURE_SCHEMA_VERSION,
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "checks": {
        "experience_no_observed_fallthrough": experience_no_observed_fallthrough,
        "skill_alias_verification": alias_verification,
        "language_values_supported": language_report["passed"],
        "text_builder_fixtures_non_empty": bool(profile_texts[0] and job_texts[0] and cv_fixture_text),
        "embedding_cache_validation_passes": cache_validation["passed"],
        "embedding_cache_mismatch_rejected": not mismatch_validation["passed"],
    },
    "report_paths": {
        "feature_quality": "reports/phase_14_feature_quality_report.json",
        "embedding_manifest": "reports/phase_14_embedding_manifest.json",
    },
}
phase_report["passed"] = all(bool(value) for value in phase_report["checks"].values())

write_json_report("phase_14_feature_quality_report.json", feature_quality_report)
write_json_report("phase_14_embedding_manifest.json", embedding_manifest)
write_json_report("phase_14_normalization_feature_builder.json", phase_report)

phase_report


{'phase_id': 'phase_14_normalization_feature_builder',
 'status': 'complete',
 'schema_version': 'normalization-feature-builder-v1',
 'generated_at': '2026-06-02T04:45:50.294317+00:00',
 'checks': {'experience_no_observed_fallthrough': True,
  'skill_alias_verification': {'ReactJS': True,
   'analisis data': True,
   'k8s': True},
  'language_values_supported': True,
  'text_builder_fixtures_non_empty': True,
  'embedding_cache_validation_passes': True,
  'embedding_cache_mismatch_rejected': True},
 'report_paths': {'feature_quality': 'reports/phase_14_feature_quality_report.json',
  'embedding_manifest': 'reports/phase_14_embedding_manifest.json'},
 'passed': True}